# Model Error Analysis

This notebook analyzes the trained model's predictions to understand **where, why, and how** it fails. The findings directly inform which post-processing steps to apply.

### Analyses
| # | Analysis | What it tells us |
|---|----------|------------------|
| 1 | Per-sample IoU distribution | Which images the model struggles with |
| 2 | Failure gallery (worst samples) | Visual patterns in failures |
| 3 | FP vs FN analysis | Is the model over- or under-predicting? |
| 4 | Performance by road coverage | Does sparsity cause failures? |
| 5 | Thin vs wide road performance | Does road width affect quality? |
| 6 | Spatial error heatmap | Where in the image do errors concentrate? |
| 7 | Calibration & reliability diagram | Can we trust the confidence scores? |
| 8 | Actionable recommendations | Which post-processing steps to apply |

---
## 1. Setup

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    PROJECT_ROOT = __import__('pathlib').Path(REPO_DIR).resolve()
else:
    PROJECT_ROOT = __import__('pathlib').Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import cv2
from tqdm import tqdm
from pathlib import Path

plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

---
## 2. Load Model & Generate Predictions

In [ ]:
# ====== SET YOUR CHECKPOINT PATH ======
CHECKPOINT = ""  # <-- SET THIS
assert CHECKPOINT, "Set CHECKPOINT path above"

state = torch.load(CHECKPOINT, map_location=device, weights_only=False)
config_dict = state.get("config", {})
model_cfg = config_dict.get("model", {})
data_cfg = config_dict.get("data", {})
norm_cfg = config_dict.get("normalization", {})
mean = norm_cfg.get("mean", [0.485, 0.456, 0.406])
std = norm_cfg.get("std", [0.229, 0.224, 0.225])
image_size = data_cfg.get("image_size", 512)

from road_segmentation.models.factory import create_model
model = create_model(
    arch=model_cfg.get("arch", "Unet"),
    encoder_name=model_cfg.get("encoder_name", "resnet34"),
    encoder_weights=None,
)
model.load_state_dict(state["model_state_dict"])
model = model.to(device).eval()
print(f"Loaded: {model_cfg.get('arch')}+{model_cfg.get('encoder_name')} | image_size={image_size}")

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_val_transform
from road_segmentation.data.dataset import RoadSegmentationDataset
from road_segmentation.training.visualization import denormalize_image
from torch.utils.data import DataLoader

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(
    pairs,
    val_ratio=data_cfg.get("val_split_ratio", 0.15),
    seed=data_cfg.get("val_split_seed", 42),
)

val_transform = get_val_transform(image_size, mean, std)
val_dataset = RoadSegmentationDataset(val_pairs, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Generating predictions for {len(val_pairs)} validation samples...")

all_probs = []
all_gts = []
all_images = []

with torch.no_grad():
    for batch in tqdm(val_loader):
        images = batch["image"].to(device)
        masks = batch["mask"]
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        for i in range(len(probs)):
            all_probs.append(probs[i, 0])
            all_gts.append(masks[i, 0].numpy())
            all_images.append(batch["image"][i])

print(f"Done: {len(all_probs)} predictions")

---
## 3. Per-Sample IoU Distribution

How consistent is the model across different images?

In [ ]:
def sample_iou(prob, gt, threshold=0.5):
    pred = (prob >= threshold).astype(bool)
    gt_b = (gt > 0).astype(bool)
    intersection = (pred & gt_b).sum()
    union = (pred | gt_b).sum()
    return intersection / max(union, 1)

def sample_metrics(prob, gt, threshold=0.5):
    pred = (prob >= threshold).astype(bool)
    gt_b = (gt > 0).astype(bool)
    tp = (pred & gt_b).sum()
    fp = (pred & ~gt_b).sum()
    fn = (~pred & gt_b).sum()
    iou = tp / max(tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    coverage = gt_b.mean() * 100
    return {"iou": iou, "precision": precision, "recall": recall, "coverage_pct": coverage,
            "tp": int(tp), "fp": int(fp), "fn": int(fn)}

# Compute per-sample metrics
sample_data = []
for i, (prob, gt) in enumerate(zip(all_probs, all_gts)):
    m = sample_metrics(prob, gt)
    m["sample_idx"] = i
    sample_data.append(m)

sample_df = pd.DataFrame(sample_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IoU distribution
axes[0].hist(sample_df["iou"], bins=50, color="#2b8cbe", edgecolor="white")
axes[0].axvline(sample_df["iou"].mean(), color="red", linestyle="--",
                label=f'Mean: {sample_df["iou"].mean():.3f}')
axes[0].axvline(sample_df["iou"].median(), color="orange", linestyle="--",
                label=f'Median: {sample_df["iou"].median():.3f}')
axes[0].set_title("Per-Sample IoU Distribution")
axes[0].set_xlabel("IoU")
axes[0].legend()

# Precision vs Recall scatter
axes[1].scatter(sample_df["recall"], sample_df["precision"], alpha=0.3, s=10)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision vs Recall (per sample)")
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)

# IoU percentiles
percentiles = [5, 10, 25, 50, 75, 90, 95]
pct_values = [sample_df["iou"].quantile(p/100) for p in percentiles]
axes[2].barh([f"p{p}" for p in percentiles], pct_values, color="#41ae76")
axes[2].set_xlabel("IoU")
axes[2].set_title("IoU Percentiles")

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Mean IoU:   {sample_df['iou'].mean():.4f}")
print(f"  Median IoU: {sample_df['iou'].median():.4f}")
print(f"  Std IoU:    {sample_df['iou'].std():.4f}")
print(f"  Worst 5%:   IoU < {sample_df['iou'].quantile(0.05):.4f}")
print(f"  Best 5%:    IoU > {sample_df['iou'].quantile(0.95):.4f}")

---
## 4. Failure Gallery — Worst Predictions

What do the model's worst predictions look like?

In [ ]:
def show_error_overlay(img_tensor, gt, prob, threshold=0.5):
    """Show image with TP=green, FP=red, FN=blue overlay."""
    img = denormalize_image(img_tensor, mean, std)
    pred = (prob >= threshold)
    gt_b = (gt > 0)

    overlay = img.astype(np.float32)
    alpha = 0.5
    tp = pred & gt_b
    fp = pred & ~gt_b
    fn = ~pred & gt_b

    overlay[tp] = overlay[tp] * (1 - alpha) + np.array([0, 255, 0]) * alpha
    overlay[fp] = overlay[fp] * (1 - alpha) + np.array([255, 0, 0]) * alpha
    overlay[fn] = overlay[fn] * (1 - alpha) + np.array([0, 0, 255]) * alpha
    return np.clip(overlay, 0, 255).astype(np.uint8)

# Show 8 worst samples
worst_idx = sample_df.nsmallest(8, "iou")["sample_idx"].values

fig, axes = plt.subplots(8, 4, figsize=(16, 32))
col_titles = ["Input", "Ground Truth", "Prediction", "Errors (TP/FP/FN)"]

for row, idx in enumerate(worst_idx):
    img = denormalize_image(all_images[idx], mean, std)
    gt = all_gts[idx]
    prob = all_probs[idx]
    pred = (prob >= 0.5)
    iou_val = sample_df.iloc[idx]["iou"]

    axes[row, 0].imshow(img)
    axes[row, 1].imshow(gt, cmap="gray")
    axes[row, 2].imshow(pred, cmap="gray")
    axes[row, 3].imshow(show_error_overlay(all_images[idx], gt, prob))

    axes[row, 0].set_ylabel(f"IoU={iou_val:.3f}", fontsize=10, fontweight="bold")
    for j in range(4):
        axes[row, j].axis("off")
        if row == 0:
            axes[row, j].set_title(col_titles[j], fontsize=11)

fig.suptitle("8 Worst Predictions (lowest IoU)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. FP vs FN Analysis

Is the model over-predicting (too many false positives) or under-predicting (too many false negatives)?

In [ ]:
total_tp = sample_df["tp"].sum()
total_fp = sample_df["fp"].sum()
total_fn = sample_df["fn"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Global FP/FN balance
labels = ["True Positive", "False Positive", "False Negative"]
values = [total_tp, total_fp, total_fn]
colors = ["#41ae76", "#ef6548", "#4292c6"]
axes[0].bar(labels, values, color=colors)
axes[0].set_title("Global Error Breakdown")
axes[0].set_ylabel("Total pixels")
for i, v in enumerate(values):
    axes[0].text(i, v + v*0.02, f"{v/1e6:.1f}M", ha="center", fontsize=10)

# Per-sample FP ratio vs FN ratio
sample_df["fp_ratio"] = sample_df["fp"] / (sample_df["fp"] + sample_df["fn"] + 1)
sample_df["fn_ratio"] = sample_df["fn"] / (sample_df["fp"] + sample_df["fn"] + 1)

axes[1].hist(sample_df["fp_ratio"], bins=40, alpha=0.6, color="#ef6548", label="FP-dominant")
axes[1].hist(sample_df["fn_ratio"], bins=40, alpha=0.6, color="#4292c6", label="FN-dominant")
axes[1].set_title("Per-Sample Error Type Distribution")
axes[1].set_xlabel("Ratio of total errors")
axes[1].legend()

plt.tight_layout()
plt.show()

fp_dominant = (sample_df["fp"] > sample_df["fn"]).sum()
fn_dominant = (sample_df["fn"] > sample_df["fp"]).sum()
print(f"FP-dominant samples: {fp_dominant} ({fp_dominant/len(sample_df)*100:.1f}%)")
print(f"FN-dominant samples: {fn_dominant} ({fn_dominant/len(sample_df)*100:.1f}%)")
print(f"\nGlobal FP/FN ratio: {total_fp/max(total_fn,1):.2f}")
print(f"  > 1 means model over-predicts (more FP than FN)")
print(f"  < 1 means model under-predicts (more FN than FP)")

---
## 6. Performance by Road Coverage

Does the model struggle more on sparse (rural) vs dense (urban) road images?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: coverage vs IoU
axes[0].scatter(sample_df["coverage_pct"], sample_df["iou"], alpha=0.3, s=15)
# Trend line
z = np.polyfit(sample_df["coverage_pct"], sample_df["iou"], 2)
x_smooth = np.linspace(0, sample_df["coverage_pct"].max(), 100)
axes[0].plot(x_smooth, np.polyval(z, x_smooth), 'r-', linewidth=2, label="Trend")
axes[0].set_xlabel("Road Coverage (%)")
axes[0].set_ylabel("IoU")
axes[0].set_title("IoU vs Road Coverage")
axes[0].legend()

# Binned performance
bins = [0, 1, 3, 5, 10, 20, 100]
labels_b = ["<1%", "1-3%", "3-5%", "5-10%", "10-20%", ">20%"]
sample_df["coverage_bin"] = pd.cut(sample_df["coverage_pct"], bins=bins, labels=labels_b)
binned = sample_df.groupby("coverage_bin", observed=True).agg(
    mean_iou=("iou", "mean"),
    count=("iou", "count"),
).reset_index()

axes[1].bar(binned["coverage_bin"].astype(str), binned["mean_iou"], color="#2b8cbe")
axes[1].set_xlabel("Road Coverage Bin")
axes[1].set_ylabel("Mean IoU")
axes[1].set_title("Mean IoU by Coverage Bin")
for i, row in binned.iterrows():
    axes[1].text(i, row["mean_iou"] + 0.01, f"n={row['count']}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

display(binned.round(4))

---
## 7. Thin vs Wide Road Performance

Does the model struggle more with thin rural roads vs wide highways?

In [ ]:
# Estimate road width per sample using distance transform
road_widths = []
for gt in all_gts:
    binary = (gt > 0).astype(np.uint8)
    if binary.sum() == 0:
        road_widths.append(0)
        continue
    dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    road_dists = dist[dist > 0]
    if len(road_dists) > 0:
        road_widths.append(2 * np.percentile(road_dists, 50))  # median width
    else:
        road_widths.append(0)

sample_df["road_width_px"] = road_widths

# Filter out zero-width samples
has_roads = sample_df[sample_df["road_width_px"] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(has_roads["road_width_px"], has_roads["iou"], alpha=0.3, s=15)
z = np.polyfit(has_roads["road_width_px"], has_roads["iou"], 2)
x_smooth = np.linspace(has_roads["road_width_px"].min(), has_roads["road_width_px"].max(), 100)
axes[0].plot(x_smooth, np.polyval(z, x_smooth), 'r-', linewidth=2, label="Trend")
axes[0].set_xlabel("Median Road Width (pixels)")
axes[0].set_ylabel("IoU")
axes[0].set_title("IoU vs Road Width")
axes[0].legend()

# Binned
width_bins = [0, 4, 8, 15, 30, 200]
width_labels = ["<4px", "4-8px", "8-15px", "15-30px", ">30px"]
has_roads = has_roads.copy()
has_roads["width_bin"] = pd.cut(has_roads["road_width_px"], bins=width_bins, labels=width_labels)
binned_w = has_roads.groupby("width_bin", observed=True).agg(
    mean_iou=("iou", "mean"),
    count=("iou", "count"),
).reset_index()

axes[1].bar(binned_w["width_bin"].astype(str), binned_w["mean_iou"], color="#7bccc4")
axes[1].set_xlabel("Road Width Bin")
axes[1].set_ylabel("Mean IoU")
axes[1].set_title("Mean IoU by Road Width")
for i, row in binned_w.iterrows():
    axes[1].text(i, row["mean_iou"] + 0.01, f"n={row['count']}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

display(binned_w.round(4))

---
## 8. Spatial Error Heatmap

Do errors concentrate at image edges, center, or uniformly?

In [ ]:
# Accumulate FP and FN maps across all samples
h, w = all_probs[0].shape
fp_heatmap = np.zeros((h, w), dtype=np.float64)
fn_heatmap = np.zeros((h, w), dtype=np.float64)

for prob, gt in zip(all_probs, all_gts):
    pred = (prob >= 0.5)
    gt_b = (gt > 0)
    fp_heatmap += (pred & ~gt_b).astype(np.float64)
    fn_heatmap += (~pred & gt_b).astype(np.float64)

fp_heatmap /= len(all_probs)
fn_heatmap /= len(all_probs)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(fp_heatmap, cmap="Reds")
axes[0].set_title("False Positive Heatmap\n(model predicts road but isn't)")
plt.colorbar(im0, ax=axes[0], label="Avg FP frequency")

im1 = axes[1].imshow(fn_heatmap, cmap="Blues")
axes[1].set_title("False Negative Heatmap\n(model misses real road)")
plt.colorbar(im1, ax=axes[1], label="Avg FN frequency")

im2 = axes[2].imshow(fp_heatmap + fn_heatmap, cmap="hot")
axes[2].set_title("Total Error Heatmap")
plt.colorbar(im2, ax=axes[2], label="Avg error frequency")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# Check if errors concentrate at edges
border = 50
center_err = (fp_heatmap + fn_heatmap)[border:-border, border:-border].mean()
edge_err = (fp_heatmap + fn_heatmap).mean() - center_err
print(f"Center error density: {center_err:.6f}")
print(f"Edge excess error:    {edge_err:.6f}")
if edge_err > center_err * 0.1:
    print("Errors are higher at image edges — consider overlapping inference or padding.")
else:
    print("Errors are distributed uniformly — no edge artifacts.")

---
## 9. Calibration Analysis

Can we trust the model's confidence scores? A well-calibrated model's predicted probability should match the actual frequency of the positive class.

In [ ]:
# Subsample pixels for efficiency (full images would be billions of pixels)
N_CALIB_SAMPLES = min(200, len(all_probs))
PIXELS_PER_SAMPLE = 10000

sampled_probs = []
sampled_gts = []

rng = np.random.RandomState(42)
for i in range(N_CALIB_SAMPLES):
    prob_flat = all_probs[i].ravel()
    gt_flat = all_gts[i].ravel()
    idx = rng.choice(len(prob_flat), size=PIXELS_PER_SAMPLE, replace=False)
    sampled_probs.extend(prob_flat[idx])
    sampled_gts.extend(gt_flat[idx])

sampled_probs = np.array(sampled_probs)
sampled_gts = (np.array(sampled_gts) > 0).astype(int)

print(f"Calibration analysis on {len(sampled_probs):,} pixels")

# Reliability diagram
n_bins = 20
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

bin_true_freq = []
bin_pred_conf = []
bin_counts = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (sampled_probs >= lo) & (sampled_probs < hi)
    if mask.sum() > 0:
        bin_true_freq.append(sampled_gts[mask].mean())
        bin_pred_conf.append(sampled_probs[mask].mean())
        bin_counts.append(mask.sum())
    else:
        bin_true_freq.append(np.nan)
        bin_pred_conf.append(np.nan)
        bin_counts.append(0)

bin_true_freq = np.array(bin_true_freq)
bin_pred_conf = np.array(bin_pred_conf)
bin_counts = np.array(bin_counts)

# ECE (Expected Calibration Error)
valid = ~np.isnan(bin_true_freq)
ece = np.sum(bin_counts[valid] * np.abs(bin_true_freq[valid] - bin_pred_conf[valid])) / bin_counts[valid].sum()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Reliability diagram
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label="Perfect calibration")
axes[0].bar(bin_centers, bin_true_freq, width=1/n_bins * 0.8, alpha=0.7, color="#2b8cbe",
            edgecolor="white", label="Model")
axes[0].set_xlabel("Predicted Confidence")
axes[0].set_ylabel("Actual Road Frequency")
axes[0].set_title(f"Reliability Diagram (ECE={ece:.4f})")
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Confidence histogram
axes[1].hist(sampled_probs, bins=50, color="#7bccc4", edgecolor="white")
axes[1].set_xlabel("Predicted Probability")
axes[1].set_ylabel("Pixel Count")
axes[1].set_title("Confidence Distribution")
axes[1].set_yscale("log")

# Calibration gap per bin
gaps = bin_true_freq - bin_pred_conf
gap_colors = ['#ef6548' if g < 0 else '#41ae76' for g in gaps]
axes[2].bar(bin_centers, gaps, width=1/n_bins * 0.8, color=gap_colors, edgecolor="white")
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel("Predicted Confidence")
axes[2].set_ylabel("Gap (actual - predicted)")
axes[2].set_title("Calibration Gap\n(positive = underconfident, negative = overconfident)")

plt.tight_layout()
plt.show()

print(f"\nExpected Calibration Error (ECE): {ece:.4f}")
if ece < 0.05:
    print("Model is well-calibrated. Confidence scores are reliable for gap bridging.")
elif ece < 0.10:
    print("Model is moderately calibrated. Consider temperature scaling for better post-processing.")
else:
    print("Model is poorly calibrated. Temperature scaling recommended before confidence-based post-processing.")

# Check over/under confidence
overconfident = (gaps[valid] < -0.05).sum()
underconfident = (gaps[valid] > 0.05).sum()
print(f"Overconfident bins: {overconfident}/{valid.sum()}")
print(f"Underconfident bins: {underconfident}/{valid.sum()}")

### Temperature Scaling (if needed)

A single-parameter fix for miscalibrated models. Finds a temperature T that rescales logits before sigmoid.

In [ ]:
from scipy.optimize import minimize_scalar

# Convert probs back to logits for temperature scaling
logits_flat = np.log(sampled_probs.clip(1e-7, 1 - 1e-7) / (1 - sampled_probs.clip(1e-7, 1 - 1e-7)))

def nll_with_temperature(T):
    """Negative log-likelihood with temperature T."""
    scaled = logits_flat / T
    probs_t = 1 / (1 + np.exp(-scaled))
    probs_t = probs_t.clip(1e-7, 1 - 1e-7)
    nll = -(sampled_gts * np.log(probs_t) + (1 - sampled_gts) * np.log(1 - probs_t)).mean()
    return nll

result = minimize_scalar(nll_with_temperature, bounds=(0.1, 10.0), method='bounded')
optimal_T = result.x

# Recompute ECE with temperature scaling
scaled_probs = 1 / (1 + np.exp(-logits_flat / optimal_T))

bin_true_scaled = []
bin_pred_scaled = []
bin_counts_scaled = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (scaled_probs >= lo) & (scaled_probs < hi)
    if mask.sum() > 0:
        bin_true_scaled.append(sampled_gts[mask].mean())
        bin_pred_scaled.append(scaled_probs[mask].mean())
        bin_counts_scaled.append(mask.sum())
    else:
        bin_true_scaled.append(np.nan)
        bin_pred_scaled.append(np.nan)
        bin_counts_scaled.append(0)

bin_true_scaled = np.array(bin_true_scaled)
bin_pred_scaled = np.array(bin_pred_scaled)
bin_counts_scaled = np.array(bin_counts_scaled)
valid_s = ~np.isnan(bin_true_scaled)
ece_scaled = np.sum(bin_counts_scaled[valid_s] * np.abs(bin_true_scaled[valid_s] - bin_pred_scaled[valid_s])) / bin_counts_scaled[valid_s].sum()

print(f"Optimal temperature: {optimal_T:.3f}")
print(f"ECE before: {ece:.4f}")
print(f"ECE after:  {ece_scaled:.4f}")
if optimal_T > 1:
    print(f"T > 1 means model is overconfident — temperature scaling softens predictions.")
else:
    print(f"T < 1 means model is underconfident — temperature scaling sharpens predictions.")

print(f"\nUse this temperature in post-processing: scale logits by 1/{optimal_T:.3f} before sigmoid.")

---
## 10. Actionable Recommendations

Based on the analysis above, which post-processing steps should we prioritize?

In [ ]:
print("Actionable Recommendations")
print("=" * 60)

# 1. FP/FN balance
fp_fn_ratio = total_fp / max(total_fn, 1)
print(f"\n1. ERROR BALANCE (FP/FN ratio = {fp_fn_ratio:.2f})")
if fp_fn_ratio > 1.5:
    print("   Model OVER-predicts. Priority: component filtering, morphological opening.")
elif fp_fn_ratio < 0.67:
    print("   Model UNDER-predicts. Priority: lower threshold, gap bridging, morphological closing.")
else:
    print("   Balanced errors. Standard pipeline applies.")

# 2. Coverage effect
sparse_iou = sample_df[sample_df["coverage_pct"] < 2]["iou"].mean()
dense_iou = sample_df[sample_df["coverage_pct"] > 10]["iou"].mean()
print(f"\n2. COVERAGE EFFECT (sparse IoU={sparse_iou:.3f}, dense IoU={dense_iou:.3f})")
if dense_iou - sparse_iou > 0.15:
    print("   Large gap — model struggles with sparse roads. Gap bridging may help.")
else:
    print("   Moderate gap — performance is relatively consistent across coverage levels.")

# 3. Road width effect
if len(has_roads) > 0:
    thin_iou = has_roads[has_roads["road_width_px"] < 6]["iou"].mean() if len(has_roads[has_roads["road_width_px"] < 6]) > 0 else 0
    wide_iou = has_roads[has_roads["road_width_px"] > 15]["iou"].mean() if len(has_roads[has_roads["road_width_px"] > 15]) > 0 else 0
    print(f"\n3. ROAD WIDTH EFFECT (thin IoU={thin_iou:.3f}, wide IoU={wide_iou:.3f})")
    if wide_iou - thin_iou > 0.15:
        print("   Thin roads are significantly harder. Boundary-weighted loss fine-tuning recommended.")
    else:
        print("   Width has moderate effect. Standard post-processing should suffice.")

# 4. Calibration
print(f"\n4. CALIBRATION (ECE={ece:.4f}, optimal T={optimal_T:.3f})")
if ece > 0.05:
    print(f"   Apply temperature scaling (T={optimal_T:.3f}) before confidence-based post-processing.")
else:
    print("   Well-calibrated. Confidence scores are reliable as-is.")

# 5. Spatial errors
print(f"\n5. SPATIAL ERRORS")
if edge_err > center_err * 0.1:
    print("   Edge artifacts detected. Consider overlapping inference with blending.")
else:
    print("   No edge artifacts. Standard inference is fine.")

print(f"\n" + "=" * 60)
print("Now run 04_postprocessing_ablation.ipynb with these findings in mind.")